# Notebook 00 — Framework Overview and Data Flow

Welcome to the **PhosCrosstalk** educational notebook suite.

PhosCrosstalk is a systems-level ODE modelling framework for phosphoproteomic  
time-series data.  It combines:

* **Phosphosite kinetics** — per-site dephosphorylation (k_off) and kinase-driven activation
* **Protein abundance dynamics** — synthesis, degradation, and crosstalk-modulated production
* **mRNA regulation** — transcription-factor network weighting of protein synthesis
* **Network priors** — PTM-database crosstalk (Cg/Cl) and kinase-substrate weights (K_site_kin)

All parameters are estimated jointly by gradient-based multi-start optimisation.


## Data Flow

```
Raw data (phospho CSV + mRNA CSV + kinase TSV + TF net)
   ↓  data_loader.py
P_data  (N×T)  phosphosite intensities
A_data  (K×T)  protein abundances
rna_matrix (K×T_rna)  mRNA levels
K_site_kin (N×M)  kinase–site weights
Cg / Cl    (N×N)  global / local PTM crosstalk
tf_prot_weights    TF→protein synthesis weights
   ↓  optimization.py / weighting.py
theta ∈ R^(2K+2+3M+N+4)   parameter vector (log-space)
bounds (xl, xu)            biologically constrained
W_data (N×T), W_prot (K×T) observation weights
   ↓  multistarts.py + diffrax ODE
theta_best,  loss = [L_phospho, L_prot, L_mrna, L_reg]
   ↓  analysis.py
Fitted time-series, rate parameters, diagnostics
   ↓  (optional) neuralODE / PINN
Refined rates or universal ODE solution
```


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load and inspect the sample data files

In [ ]:
# Phosphoproteomic data (protein rows + phosphosite rows)
df_pp = pd.read_csv(SAMPLE_DIR / "protephospho.csv")
print("protephospho.csv — shape:", df_pp.shape)
df_pp.head(6)


In [ ]:
# mRNA data
df_rna = pd.read_csv(SAMPLE_DIR / "mrna.csv")
print("mrna.csv — shape:", df_rna.shape)
df_rna.head()


In [ ]:
# Kinase–site weights
df_kin = pd.read_csv(SAMPLE_DIR / "kinase_sites.tsv", sep="\t")
print("kinase_sites.tsv — shape:", df_kin.shape)
df_kin.head()


In [ ]:
# TF–mRNA weights
df_tf = pd.read_csv(SAMPLE_DIR / "tf_mrna.csv")
print("tf_mrna.csv — shape:", df_tf.shape)
df_tf.head()


## Instantiate ModelDims from the sample data

In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import load_site_data, load_kinase_site_matrix

timepoints = list(range(1, 15))   # 14 time points x1..x14

sites, proteins, site_prot_idx, positions, t, Y, A_data, A_proteins = \
    load_site_data(SAMPLE_DIR / "protephospho.csv", timepoints)

K_site_kin, kinases = load_kinase_site_matrix(SAMPLE_DIR / "kinase_sites.tsv", sites)

K = len(proteins)   # number of proteins
M = len(kinases)    # number of kinases
N = len(sites)      # number of phosphosites

dims = ModelDims.set_dims(K, M, N)
print(f"ModelDims  K={dims.K}  M={dims.M}  N={dims.N}")
print(f"  K = {dims.K} proteins  : {proteins}")
print(f"  M = {dims.M} kinases   : {kinases}")
print(f"  N = {dims.N} phosphosites: {sites}")


### Biological meaning of K, M, N

| Symbol | Count | Meaning |
|--------|-------|---------|
| K | 3 | Receptor-tyrosine kinase proteins tracked by abundance |
| M | 2 | Kinases with known substrate-site weights (EGFR, MET) |
| N | 9 | Phosphosites whose intensities are measured over time |

The **theta** vector has dimension `2K + 2 + 3M + N + 4` = **`2·3 + 2 + 3·2 + 9 + 4 = 25`**.


## Module summary

In [ ]:
modules = {
    "phoscrosstalk.config":        "ModelDims, load_config, validate_config",
    "phoscrosstalk.data_loader":   "load_site_data, load_rna_data, load_kinase_site_matrix, load_tf_network",
    "phoscrosstalk.weighting":     "build_weight_matrices",
    "phoscrosstalk.optimization":  "create_bounds, build_parameter_labels",
    "phoscrosstalk.mechanisms":    "decode_theta, make_rhs",
    "phoscrosstalk.derived_rates": "make_k_act_fn, make_s_prod_fn",
    "phoscrosstalk.simulation":    "simulate",
}
df_mod = pd.DataFrame(list(modules.items()), columns=["Module", "Key exports"])
print(df_mod.to_string(index=False))
